# 🚀 V8 News Scraper - Parallel Processing Architecture

**Version:** 8.0  
**Key Features:**
- ⚡ Parallel source processing (ThreadPoolExecutor)
- 📁 Sources loaded from JSON configuration file
- 📊 tqdm progress bars for all operations
- 🔇 Silent fail with error logging (no crashes)
- 📄 Clean CSV output

**Architecture:**
- Single HTTPClient with session reuse
- 4-method waterfall: RSS → API → HTML → Homepage
- Composable filter chain
- Auto-detection of optimal worker count

In [1]:
# ============================================================================
# CELL 1: IMPORTS
# ============================================================================
import json
import re
import os
import logging
import hashlib
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field, asdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urlparse
import time
import warnings
warnings.filterwarnings('ignore')

# Third-party imports
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import feedparser
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm
from newspaper import Article, Config as NewspaperConfig

# Setup logging (silent mode - only to file)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('scraper_v8.log', mode='w'),
    ]
)
logger = logging.getLogger(__name__)

print("✅ All imports loaded successfully")

✅ All imports loaded successfully


In [2]:
# ============================================================================
# CELL 2: CONFIGURATION (User-Editable)
# ============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# SOURCES - Load from JSON file
# ─────────────────────────────────────────────────────────────────────────────
SOURCES_FILE = "sources.json"

with open(SOURCES_FILE, 'r') as f:
    sources_data = json.load(f)
    SOURCES = [(s['name'], s['url']) for s in sources_data['sources']]

print(f"📰 Loaded {len(SOURCES)} sources from {SOURCES_FILE}")

# ─────────────────────────────────────────────────────────────────────────────
# SEARCH KEYWORDS - Articles must contain at least one
# ─────────────────────────────────────────────────────────────────────────────
KEYWORDS = [
    # AI & Machine Learning
    "artificial intelligence", "machine learning", "deep learning", "neural network",
    "generative ai", "large language model", "llm", "chatgpt", "gpt-4", "gpt-5",
    "claude", "gemini", "copilot", "midjourney", "stable diffusion", "dall-e",
    "openai", "anthropic", "google ai", "meta ai", "microsoft ai",
    
    # Semiconductors & Hardware
    "semiconductor", "chip", "nvidia", "amd", "intel", "tsmc", "gpu", "cpu", "tpu",
    "arm", "qualcomm", "asml", "samsung semiconductor",
    
    # Tech Industry
    "tech industry", "silicon valley", "startup", "venture capital", "ipo",
    "big tech", "faang", "tech layoff", "tech regulation",
    
    # Energy & Climate
    "renewable energy", "solar", "wind energy", "battery", "ev", "electric vehicle",
    "clean energy", "climate tech", "carbon capture", "nuclear energy", "fusion",
    
    # Policy & Regulation
    "ai regulation", "tech policy", "antitrust", "data privacy", "gdpr",
    "ai safety", "ai ethics", "ai governance"
]

# ─────────────────────────────────────────────────────────────────────────────
# SCRAPING PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # Time settings
    "date_range_days": 14,          # Only articles from last N days (changed to 14)
    "timeout_seconds": 10,          # Request timeout
    "retry_count": 1,               # Retries on failure
    
    # Parallelism settings (auto-detect or manual)
    "max_workers": None,            # None = auto-detect based on CPU
    "article_workers": 8,           # Workers for fetching article content
    
    # Discovery paths (simplified for speed)
    "rss_paths": [
        "/feed", "/rss", "/feeds/all.rss", "/rss.xml", "/feed.xml",
        "/atom.xml", "/index.xml", "/news/feed", "/blog/feed"
    ],
    "api_paths": [
        "/api/articles", "/api/posts", "/api/news", "/api/v1/articles"
    ],
    
    # Quality thresholds
    "min_content_length": 200,      # Minimum article text length
    "min_title_length": 10,         # Minimum title length
    
    # Time budgets per method (seconds)
    "time_budget_rss": 30,
    "time_budget_api": 20,
    "time_budget_html": 60,
    "time_budget_homepage": 90,
    
    # Output
    "output_file": "scraped_news_v8.csv",
    "telemetry_file": "scraping_telemetry_v8.csv"
}

# Auto-detect workers based on CPU
if CONFIG["max_workers"] is None:
    CONFIG["max_workers"] = min(os.cpu_count() * 2, 16)

print(f"⚙️  Config: {CONFIG['max_workers']} workers, {CONFIG['date_range_days']} days, {CONFIG['timeout_seconds']}s timeout")

📰 Loaded 50 sources from sources.json
⚙️  Config: 16 workers, 14 days, 10s timeout


In [3]:
# ============================================================================
# CELL 3: CORE UTILITIES
# ============================================================================

@dataclass
class ScrapedArticle:
    """Data class for a scraped article."""
    title: str
    url: str
    published_date: Optional[str] = None
    content: str = ""
    author: str = ""  # Added author field
    source_name: str = ""
    source_url: str = ""
    method: str = ""
    keywords_matched: List[str] = field(default_factory=list)
    scrape_timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

class HTTPClient:
    """Reusable HTTP client with session management."""
    
    def __init__(self, timeout: int = 10, retries: int = 1):
        self.timeout = timeout
        self.session = requests.Session()
        
        # Configure retry strategy
        retry_strategy = Retry(
            total=retries,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
        )
        adapter = HTTPAdapter(max_retries=retry_strategy, pool_connections=20, pool_maxsize=20)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
        
        # Default headers
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
        })
    
    def get(self, url: str, **kwargs) -> Optional[requests.Response]:
        """Make GET request with error handling (silent fail)."""
        try:
            response = self.session.get(url, timeout=self.timeout, **kwargs)
            response.raise_for_status()
            return response
        except Exception as e:
            logger.debug(f"Request failed for {url}: {e}")
            return None
    
    def get_json(self, url: str) -> Optional[Dict]:
        """GET request expecting JSON response."""
        try:
            response = self.get(url, headers={'Accept': 'application/json'})
            if response:
                return response.json()
        except Exception as e:
            logger.debug(f"JSON parse failed for {url}: {e}")
        return None

# Global HTTP client instance
http = HTTPClient(timeout=CONFIG["timeout_seconds"], retries=CONFIG["retry_count"])

def parse_date(date_input: Any) -> Optional[datetime]:
    """Parse various date formats to datetime object."""
    if date_input is None:
        return None
    
    # Handle feedparser time_struct
    if hasattr(date_input, 'tm_year'):
        try:
            return datetime(*date_input[:6])
        except:
            return None
    
    # Handle datetime objects
    if isinstance(date_input, datetime):
        return date_input
    
    # Handle string formats
    if isinstance(date_input, str):
        date_formats = [
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%dT%H:%M:%SZ",
            "%Y-%m-%dT%H:%M:%S%z",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d",
            "%d %b %Y",
            "%B %d, %Y",
            "%b %d, %Y",
        ]
        for fmt in date_formats:
            try:
                # Remove timezone info for parsing
                clean_date = re.sub(r'[+-]\d{2}:\d{2}$', '', date_input)
                clean_date = clean_date.replace('Z', '').strip()
                return datetime.strptime(clean_date, fmt)
            except:
                continue
    return None

def strip_html(text: str) -> str:
    """Strip HTML tags from text using BeautifulSoup."""
    if not text:
        return ""
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')

def clean_text(text: str) -> str:
    """Clean and normalize text content (strips HTML first)."""
    if not text:
        return ""
    # FIRST: Strip HTML tags
    text = strip_html(text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove common boilerplate patterns
    patterns_to_remove = [
        r'Share this article.*',
        r'Subscribe to.*newsletter',
        r'Sign up for.*',
        r'Follow us on.*',
        r'Read more:.*',
        r'Related articles.*',
        r'Advertisement',
        r'ADVERTISEMENT',
    ]
    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()

def is_within_date_range(date: Optional[datetime], days: int) -> bool:
    """Check if date is within the specified range."""
    if date is None:
        return True  # Include articles with unknown dates
    cutoff = datetime.now() - timedelta(days=days)
    return date >= cutoff

def matches_keywords(text: str, keywords: List[str]) -> List[str]:
    """Find which keywords match in the text (using word boundaries)."""
    if not text:
        return []
    text_lower = text.lower()
    matched = []
    for kw in keywords:
        # Use word boundaries to avoid partial matches (e.g., "ev" matching "every")
        pattern = rf'\b{re.escape(kw.lower())}\b'
        if re.search(pattern, text_lower):
            matched.append(kw)
    return matched

def generate_article_hash(url: str, title: str) -> str:
    """Generate unique hash for deduplication."""
    content = f"{url.lower().strip()}{title.lower().strip()}"
    return hashlib.md5(content.encode()).hexdigest()

print("✅ Core utilities loaded (with HTML stripping & word-boundary keyword matching)")

✅ Core utilities loaded (with HTML stripping & word-boundary keyword matching)


In [4]:
# ============================================================================
# CELL 4: DISCOVERY MODULE
# ============================================================================

def discover_rss_feed(base_url: str, paths: List[str]) -> Optional[str]:
    """Discover RSS feed URL for a website."""
    for path in paths:
        feed_url = urljoin(base_url, path)
        response = http.get(feed_url)
        if response and response.status_code == 200:
            content_type = response.headers.get('content-type', '').lower()
            if 'xml' in content_type or 'rss' in content_type:
                # Verify it's actually a valid feed
                feed = feedparser.parse(response.content)
                if feed.entries:
                    logger.info(f"Found RSS feed: {feed_url}")
                    return feed_url
    
    # Try to find RSS link in HTML
    response = http.get(base_url)
    if response:
        soup = BeautifulSoup(response.text, 'html.parser')
        rss_link = soup.find('link', type=re.compile(r'(rss|atom)', re.I))
        if rss_link and rss_link.get('href'):
            return urljoin(base_url, rss_link['href'])
    
    return None

def discover_api_endpoint(base_url: str, paths: List[str]) -> Optional[str]:
    """Discover API endpoint for a website."""
    for path in paths:
        api_url = urljoin(base_url, path)
        data = http.get_json(api_url)
        if data and isinstance(data, (list, dict)):
            # Check if it looks like article data
            items = data if isinstance(data, list) else data.get('articles', data.get('posts', data.get('items', [])))
            if items and len(items) > 0:
                logger.info(f"Found API endpoint: {api_url}")
                return api_url
    return None

def discover_sitemap(base_url: str) -> Optional[str]:
    """Discover sitemap URL."""
    sitemap_paths = ['/sitemap.xml', '/sitemap_index.xml', '/news-sitemap.xml']
    for path in sitemap_paths:
        sitemap_url = urljoin(base_url, path)
        response = http.get(sitemap_url)
        if response and 'xml' in response.headers.get('content-type', '').lower():
            return sitemap_url
    return None

print("✅ Discovery module loaded")

✅ Discovery module loaded


In [5]:
# ============================================================================
# CELL 5: SCRAPERS
# ============================================================================

# Newspaper3k configuration
newspaper_config = NewspaperConfig()
newspaper_config.browser_user_agent = http.session.headers['User-Agent']
newspaper_config.request_timeout = CONFIG["timeout_seconds"]
newspaper_config.fetch_images = False
newspaper_config.memoize_articles = False

def fetch_article_content(url: str) -> Tuple[str, str, str, Optional[datetime]]:
    """
    Fetch full article content using newspaper3k. 
    Returns (title, content, author, publish_date).
    """
    try:
        article = Article(url, config=newspaper_config)
        article.download()
        article.parse()
        
        title = article.title or ""
        content = clean_text(article.text or "")
        
        # Extract author (newspaper3k returns list of authors)
        author = ""
        if article.authors:
            author = ", ".join(article.authors)
        
        # Extract publish date
        pub_date = article.publish_date if article.publish_date else None
        
        return title, content, author, pub_date
    except Exception as e:
        logger.debug(f"Failed to fetch article {url}: {e}")
        return "", "", "", None

def scrape_rss(feed_url: str, source_name: str, source_url: str, 
               time_budget: float) -> List[ScrapedArticle]:
    """Scrape articles from RSS feed."""
    articles = []
    start_time = time.time()
    
    try:
        feed = feedparser.parse(feed_url)
        entries = feed.entries[:30]  # Limit entries
        
        for entry in entries:
            if time.time() - start_time > time_budget:
                logger.info(f"RSS time budget exceeded for {source_name}")
                break
            
            title = entry.get('title', '')
            url = entry.get('link', '')
            
            if not url or not title:
                continue
            
            # Parse date
            pub_date = parse_date(entry.get('published_parsed') or entry.get('updated_parsed'))
            
            # Get author from RSS entry
            author = entry.get('author', '') or entry.get('creator', '')
            
            # Get content from feed - clean HTML from RSS summary
            raw_content = entry.get('summary', '') or entry.get('description', '')
            content = clean_text(raw_content)  # clean_text now strips HTML
            
            # If content is too short, fetch full article
            if len(content) < CONFIG["min_content_length"]:
                _, fetched_content, fetched_author, fetched_date = fetch_article_content(url)
                if fetched_content:
                    content = fetched_content
                if fetched_author and not author:
                    author = fetched_author
                if fetched_date and not pub_date:
                    pub_date = fetched_date
            
            articles.append(ScrapedArticle(
                title=title,
                url=url,
                published_date=pub_date.isoformat() if pub_date else None,
                content=content,
                author=author,
                source_name=source_name,
                source_url=source_url,
                method="rss"
            ))
    except Exception as e:
        logger.warning(f"RSS scrape failed for {source_name}: {e}")
    
    return articles

def scrape_api(api_url: str, source_name: str, source_url: str,
               time_budget: float) -> List[ScrapedArticle]:
    """Scrape articles from API endpoint."""
    articles = []
    start_time = time.time()
    
    try:
        data = http.get_json(api_url)
        if not data:
            return []
        
        # Handle different API response formats
        items = data if isinstance(data, list) else data.get('articles', data.get('posts', data.get('items', data.get('data', []))))
        
        for item in items[:30]:
            if time.time() - start_time > time_budget:
                break
            
            title = item.get('title') or item.get('headline', '')
            url = item.get('url') or item.get('link', '')
            
            if not url or not title:
                continue
            
            if not url.startswith('http'):
                url = urljoin(source_url, url)
            
            pub_date = parse_date(item.get('published_at') or item.get('date') or item.get('publishedAt'))
            author = item.get('author') or item.get('author_name', '')
            content = item.get('content') or item.get('body') or item.get('description', '')
            
            if len(content) < CONFIG["min_content_length"]:
                _, fetched_content, fetched_author, fetched_date = fetch_article_content(url)
                if fetched_content:
                    content = fetched_content
                if fetched_author and not author:
                    author = fetched_author
                if fetched_date and not pub_date:
                    pub_date = fetched_date
            
            articles.append(ScrapedArticle(
                title=title,
                url=url,
                published_date=pub_date.isoformat() if pub_date else None,
                content=clean_text(content),
                author=author,
                source_name=source_name,
                source_url=source_url,
                method="api"
            ))
    except Exception as e:
        logger.warning(f"API scrape failed for {source_name}: {e}")
    
    return articles

def scrape_html(base_url: str, source_name: str, time_budget: float) -> List[ScrapedArticle]:
    """Scrape articles by finding article links in HTML."""
    articles = []
    start_time = time.time()
    
    try:
        response = http.get(base_url)
        if not response:
            return []
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find article links (common patterns)
        article_selectors = [
            'article a[href]',
            '.article a[href]',
            '.post a[href]',
            '.story a[href]',
            'a[href*="/article/"]',
            'a[href*="/news/"]',
            'a[href*="/story/"]',
            'a[href*="/post/"]',
        ]
        
        seen_urls = set()
        links = []
        
        for selector in article_selectors:
            for link in soup.select(selector):
                href = link.get('href', '')
                if href and href not in seen_urls:
                    full_url = urljoin(base_url, href)
                    if urlparse(full_url).netloc == urlparse(base_url).netloc:
                        seen_urls.add(href)
                        links.append(full_url)
        
        # Fetch articles in parallel
        def fetch_one(url):
            if time.time() - start_time > time_budget:
                return None
            title, content, author, pub_date = fetch_article_content(url)
            if title and len(content) >= CONFIG["min_content_length"]:
                return ScrapedArticle(
                    title=title,
                    url=url,
                    published_date=pub_date.isoformat() if pub_date else None,
                    content=content,
                    author=author,
                    source_name=source_name,
                    source_url=base_url,
                    method="html"
                )
            return None
        
        with ThreadPoolExecutor(max_workers=CONFIG["article_workers"]) as executor:
            futures = [executor.submit(fetch_one, url) for url in links[:20]]
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        articles.append(result)
                except:
                    pass
    except Exception as e:
        logger.warning(f"HTML scrape failed for {source_name}: {e}")
    
    return articles

def scrape_homepage(base_url: str, source_name: str, time_budget: float) -> List[ScrapedArticle]:
    """Scrape articles directly from homepage links."""
    articles = []
    start_time = time.time()
    
    try:
        response = http.get(base_url)
        if not response:
            return []
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all links and filter for likely articles
        all_links = soup.find_all('a', href=True)
        article_urls = []
        
        for link in all_links:
            href = link.get('href', '')
            full_url = urljoin(base_url, href)
            
            # Filter for article-like URLs
            parsed = urlparse(full_url)
            if parsed.netloc != urlparse(base_url).netloc:
                continue
            
            path = parsed.path.lower()
            # Skip non-article paths
            if any(skip in path for skip in ['/tag/', '/category/', '/author/', '/login', '/signup', '/subscribe']):
                continue
            # Look for article-like paths
            if any(pattern in path for pattern in ['/20', '/article', '/news/', '/story/', '/post/']):
                if full_url not in article_urls:
                    article_urls.append(full_url)
        
        # Fetch articles in parallel
        def fetch_one(url):
            if time.time() - start_time > time_budget:
                return None
            title, content, author, pub_date = fetch_article_content(url)
            if title and len(content) >= CONFIG["min_content_length"]:
                return ScrapedArticle(
                    title=title,
                    url=url,
                    published_date=pub_date.isoformat() if pub_date else None,
                    content=content,
                    author=author,
                    source_name=source_name,
                    source_url=base_url,
                    method="homepage"
                )
            return None
        
        with ThreadPoolExecutor(max_workers=CONFIG["article_workers"]) as executor:
            futures = [executor.submit(fetch_one, url) for url in article_urls[:25]]
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        articles.append(result)
                except:
                    pass
    except Exception as e:
        logger.warning(f"Homepage scrape failed for {source_name}: {e}")
    
    return articles

print("✅ Scraper functions loaded (with author & date extraction)")

✅ Scraper functions loaded (with author & date extraction)


In [6]:
# ============================================================================
# CELL 6: FILTERS
# ============================================================================

def filter_by_date(articles: List[ScrapedArticle], days: int) -> List[ScrapedArticle]:
    """Filter articles by date range."""
    filtered = []
    for article in articles:
        pub_date = parse_date(article.published_date)
        if is_within_date_range(pub_date, days):
            filtered.append(article)
    return filtered

def filter_by_keywords(articles: List[ScrapedArticle], keywords: List[str]) -> List[ScrapedArticle]:
    """Filter articles that match at least one keyword."""
    filtered = []
    for article in articles:
        search_text = f"{article.title} {article.content}"
        matched = matches_keywords(search_text, keywords)
        if matched:
            article.keywords_matched = matched
            filtered.append(article)
    return filtered

def filter_by_quality(articles: List[ScrapedArticle], 
                      min_content_length: int = 200,
                      min_title_length: int = 10) -> List[ScrapedArticle]:
    """Filter articles by quality thresholds."""
    return [
        a for a in articles
        if len(a.content) >= min_content_length and len(a.title) >= min_title_length
    ]

def deduplicate(articles: List[ScrapedArticle]) -> List[ScrapedArticle]:
    """Remove duplicate articles based on URL and title."""
    seen_hashes = set()
    unique = []
    
    for article in articles:
        article_hash = generate_article_hash(article.url, article.title)
        if article_hash not in seen_hashes:
            seen_hashes.add(article_hash)
            unique.append(article)
    
    return unique

def apply_filters(articles: List[ScrapedArticle], 
                  date_range_days: int = 7,
                  keywords: List[str] = None,
                  min_content: int = 200) -> List[ScrapedArticle]:
    """Apply all filters in sequence."""
    # Date filter
    articles = filter_by_date(articles, date_range_days)
    
    # Keyword filter
    if keywords:
        articles = filter_by_keywords(articles, keywords)
    
    # Quality filter
    articles = filter_by_quality(articles, min_content_length=min_content)
    
    # Deduplication
    articles = deduplicate(articles)
    
    return articles

print("✅ Filter functions loaded")

✅ Filter functions loaded


In [7]:
# ============================================================================
# CELL 7: ORCHESTRATOR
# ============================================================================

@dataclass
class SourceResult:
    """Result from scraping a single source."""
    source_name: str
    source_url: str
    method_used: str
    articles_found: int
    articles_after_filter: int
    elapsed_seconds: float
    success: bool
    error: Optional[str] = None

def scrape_source(source_name: str, source_url: str) -> Tuple[List[ScrapedArticle], SourceResult]:
    """
    Scrape a single source using waterfall strategy: RSS → API → HTML → Homepage.
    Returns filtered articles and telemetry.
    """
    start_time = time.time()
    articles = []
    method_used = "none"
    error = None
    
    try:
        # Try RSS first (fastest)
        rss_url = discover_rss_feed(source_url, CONFIG["rss_paths"])
        if rss_url:
            articles = scrape_rss(rss_url, source_name, source_url, CONFIG["time_budget_rss"])
            if articles:
                method_used = "rss"
                logger.info(f"[RSS] {source_name}: {len(articles)} articles")
        
        # Try API if RSS failed
        if not articles:
            api_url = discover_api_endpoint(source_url, CONFIG["api_paths"])
            if api_url:
                articles = scrape_api(api_url, source_name, source_url, CONFIG["time_budget_api"])
                if articles:
                    method_used = "api"
                    logger.info(f"[API] {source_name}: {len(articles)} articles")
        
        # Try HTML parsing
        if not articles:
            articles = scrape_html(source_url, source_name, CONFIG["time_budget_html"])
            if articles:
                method_used = "html"
                logger.info(f"[HTML] {source_name}: {len(articles)} articles")
        
        # Last resort: homepage scraping
        if not articles:
            articles = scrape_homepage(source_url, source_name, CONFIG["time_budget_homepage"])
            if articles:
                method_used = "homepage"
                logger.info(f"[Homepage] {source_name}: {len(articles)} articles")
        
    except Exception as e:
        error = str(e)
        logger.error(f"Error scraping {source_name}: {e}")
    
    # Apply filters
    raw_count = len(articles)
    articles = apply_filters(
        articles,
        date_range_days=CONFIG["date_range_days"],
        keywords=KEYWORDS,
        min_content=CONFIG["min_content_length"]
    )
    
    elapsed = time.time() - start_time
    
    result = SourceResult(
        source_name=source_name,
        source_url=source_url,
        method_used=method_used,
        articles_found=raw_count,
        articles_after_filter=len(articles),
        elapsed_seconds=round(elapsed, 2),
        success=len(articles) > 0,
        error=error
    )
    
    return articles, result

def scrape_all_sources_parallel(sources: List[Tuple[str, str]], 
                                  max_workers: int) -> Tuple[List[ScrapedArticle], List[SourceResult]]:
    """
    Scrape all sources in parallel using ThreadPoolExecutor.
    Returns all articles and telemetry for each source.
    """
    all_articles = []
    all_results = []
    
    print(f"\n🚀 Starting parallel scrape with {max_workers} workers...")
    print(f"📰 Processing {len(sources)} sources\n")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_source = {
            executor.submit(scrape_source, name, url): (name, url)
            for name, url in sources
        }
        
        # Process completed tasks with progress bar
        with tqdm(total=len(sources), desc="Scraping sources", unit="source") as pbar:
            for future in as_completed(future_to_source):
                source_name, source_url = future_to_source[future]
                try:
                    articles, result = future.result()
                    all_articles.extend(articles)
                    all_results.append(result)
                    
                    # Update progress bar description
                    status = "✓" if result.success else "✗"
                    pbar.set_postfix_str(f"{status} {source_name[:20]}: {result.articles_after_filter} articles")
                except Exception as e:
                    # Silent fail - log error but continue
                    logger.error(f"Failed to scrape {source_name}: {e}")
                    all_results.append(SourceResult(
                        source_name=source_name,
                        source_url=source_url,
                        method_used="none",
                        articles_found=0,
                        articles_after_filter=0,
                        elapsed_seconds=0,
                        success=False,
                        error=str(e)
                    ))
                
                pbar.update(1)
    
    # Final deduplication across all sources
    all_articles = deduplicate(all_articles)
    
    return all_articles, all_results

print("✅ Orchestrator loaded")

✅ Orchestrator loaded


In [8]:
# ============================================================================
# CELL 8: MAIN EXECUTION
# ============================================================================

print("=" * 70)
print("🚀 V8 NEWS SCRAPER - PARALLEL EXECUTION")
print("=" * 70)
print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📰 Sources: {len(SOURCES)}")
print(f"🔑 Keywords: {len(KEYWORDS)}")
print(f"📅 Date range: {CONFIG['date_range_days']} days")
print(f"⚡ Workers: {CONFIG['max_workers']}")
print("=" * 70)

# Run the parallel scraper
scrape_start = time.time()

all_articles, telemetry = scrape_all_sources_parallel(
    sources=SOURCES,
    max_workers=CONFIG["max_workers"]
)

total_time = time.time() - scrape_start

print("\n" + "=" * 70)
print("✅ SCRAPING COMPLETE")
print("=" * 70)
print(f"⏱️  Total time: {total_time/60:.1f} minutes ({total_time:.0f} seconds)")
print(f"📊 Articles collected: {len(all_articles)}")
print(f"✓  Successful sources: {sum(1 for r in telemetry if r.success)}/{len(telemetry)}")
print("=" * 70)

🚀 V8 NEWS SCRAPER - PARALLEL EXECUTION
⏰ Started at: 2025-12-15 02:54:27
📰 Sources: 50
🔑 Keywords: 62
📅 Date range: 14 days
⚡ Workers: 16

🚀 Starting parallel scrape with 16 workers...
📰 Processing 50 sources



Scraping sources:   0%|          | 0/50 [00:00<?, ?source/s]


✅ SCRAPING COMPLETE
⏱️  Total time: 2.9 minutes (171 seconds)
📊 Articles collected: 182
✓  Successful sources: 29/50


In [9]:
# ============================================================================
# CELL 9: EXPORT TO CSV
# ============================================================================

import hashlib

# Convert articles to DataFrame
articles_data = [asdict(article) for article in all_articles]
df_articles = pd.DataFrame(articles_data)

# Convert keywords_matched list to string for CSV
if 'keywords_matched' in df_articles.columns:
    df_articles['keywords_matched'] = df_articles['keywords_matched'].apply(lambda x: ', '.join(x) if x else '')

# Map to V7 schema for compatibility
# source, headline, author, url, published, matched_keywords, content_snippet, url_hash, full_content, method, scraped_at
df_export = pd.DataFrame({
    'source': df_articles['source_name'],
    'headline': df_articles['title'],
    'author': df_articles['author'] if 'author' in df_articles.columns else '',  # Now uses extracted author
    'url': df_articles['url'],
    'published': df_articles['published_date'],
    'matched_keywords': df_articles['keywords_matched'],
    'content_snippet': df_articles['content'].str[:200],  # First 200 chars
    'url_hash': df_articles['url'].apply(lambda x: hashlib.md5(x.encode()).hexdigest()[:8]),
    'full_content': df_articles['content'],
    'method': df_articles['method'],
    'scraped_at': df_articles['scrape_timestamp']
})

# Save articles CSV with V7 schema
df_export.to_csv(CONFIG["output_file"], index=False, encoding='utf-8')
print(f"✅ Articles saved to: {CONFIG['output_file']}")
print(f"   • Total rows: {len(df_export)}")
print(f"   • Columns: {list(df_export.columns)}")

# Save telemetry CSV
telemetry_data = [asdict(r) for r in telemetry]
df_telemetry = pd.DataFrame(telemetry_data)
df_telemetry.to_csv(CONFIG["telemetry_file"], index=False, encoding='utf-8')
print(f"✅ Telemetry saved to: {CONFIG['telemetry_file']}")
print(f"   • Total sources: {len(df_telemetry)}")

# Preview
print("\n📰 Sample articles:")
if not df_export.empty:
    display(df_export[['headline', 'author', 'source', 'method', 'published']].head(10))

✅ Articles saved to: scraped_news_v8.csv
   • Total rows: 182
   • Columns: ['source', 'headline', 'author', 'url', 'published', 'matched_keywords', 'content_snippet', 'url_hash', 'full_content', 'method', 'scraped_at']
✅ Telemetry saved to: scraping_telemetry_v8.csv
   • Total sources: 50

📰 Sample articles:


,headline,author,source,method,published
0,Is ChatGPT’s New Shopping Research Solving a P...,Viviane Mendes,The Next Web,rss,2025-12-11T22:37:55
1,"The Download: solar geoengineering’s future, a...",Rhiannon Williams,MIT Technology Review,rss,2025-12-11T13:10:00
2,Solar geoengineering startups are getting serious,Casey Crownhart,MIT Technology Review,rss,2025-12-11T11:00:00
3,The Download: a controversial proposal to solv...,Charlotte Jee,MIT Technology Review,rss,2025-12-10T13:14:00
4,How one controversial startup hopes to cool th...,James Temple,MIT Technology Review,rss,2025-12-10T10:00:00
5,The Download: a peek at AI’s future,Charlotte Jee,MIT Technology Review,rss,2025-12-09T13:10:00
6,Why most enterprise AI coding pilots underperf...,,VentureBeat,rss,2025-12-13T20:00:00
7,Ai2's new Olmo 3.1 extends reinforcement learn...,,VentureBeat,rss,2025-12-12T05:00:00
8,Google’s new framework helps AI agents spend t...,bendee983@gmail.com (Ben Dickson),VentureBeat,rss,2025-12-12T00:00:00
9,"GPT-5.2 first impressions: a powerful update, ...",carl.franzen@venturebeat.com (Carl Franzen),VentureBeat,rss,2025-12-11T23:26:00


In [10]:
# ============================================================================
# CELL 10: ANALYTICS & SUMMARY
# ============================================================================

print("=" * 70)
print("📊 SCRAPING ANALYTICS")
print("=" * 70)

# Method breakdown
print("\n📈 Articles by Method:")
method_counts = df_articles.groupby('method').size().sort_values(ascending=False)
for method, count in method_counts.items():
    pct = count / len(df_articles) * 100 if len(df_articles) > 0 else 0
    print(f"   • {method.upper():10} : {count:4} articles ({pct:.1f}%)")

# Source success breakdown
print("\n✅ Source Success Rate:")
successful = sum(1 for r in telemetry if r.success)
failed = len(telemetry) - successful
print(f"   • Successful: {successful}/{len(telemetry)} ({successful/len(telemetry)*100:.1f}%)")
print(f"   • Failed:     {failed}/{len(telemetry)} ({failed/len(telemetry)*100:.1f}%)")

# Method usage by sources
print("\n🔧 Scraping Methods Used:")
method_usage = df_telemetry.groupby('method_used').size().sort_values(ascending=False)
for method, count in method_usage.items():
    print(f"   • {method:10} : {count} sources")

# Top performing sources
print("\n🏆 Top 10 Sources by Article Count:")
top_sources = df_telemetry.nlargest(10, 'articles_after_filter')[['source_name', 'articles_after_filter', 'method_used', 'elapsed_seconds']]
display(top_sources)

# Slowest sources
print("\n🐢 Slowest Sources:")
slow_sources = df_telemetry.nlargest(5, 'elapsed_seconds')[['source_name', 'elapsed_seconds', 'method_used']]
display(slow_sources)

# Failed sources
failed_sources = df_telemetry[~df_telemetry['success']]
if len(failed_sources) > 0:
    print(f"\n❌ Failed Sources ({len(failed_sources)}):")
    for _, row in failed_sources.iterrows():
        print(f"   • {row['source_name']}: {row.get('error', 'Unknown error')[:50]}")

# Timing summary
print("\n⏱️ Timing Summary:")
print(f"   • Total scraping time: {total_time/60:.1f} minutes")
print(f"   • Average per source: {df_telemetry['elapsed_seconds'].mean():.1f}s")
print(f"   • Median per source: {df_telemetry['elapsed_seconds'].median():.1f}s")
print(f"   • Sources processed in parallel: {CONFIG['max_workers']} at a time")

# Keywords matched
if 'keywords_matched' in df_articles.columns:
    print("\n🔑 Top Keywords Matched:")
    all_keywords = []
    for kw_str in df_articles['keywords_matched'].dropna():
        if kw_str:
            all_keywords.extend([k.strip() for k in kw_str.split(',')])
    if all_keywords:
        from collections import Counter
        kw_counts = Counter(all_keywords).most_common(10)
        for kw, count in kw_counts:
            print(f"   • {kw}: {count}")

print("\n" + "=" * 70)
print("🎉 V8 Scraper Complete!")
print("=" * 70)

📊 SCRAPING ANALYTICS

📈 Articles by Method:
   • RSS        :  158 articles (86.8%)
   • HOMEPAGE   :   14 articles (7.7%)
   • HTML       :   10 articles (5.5%)

✅ Source Success Rate:
   • Successful: 29/50 (58.0%)
   • Failed:     21/50 (42.0%)

🔧 Scraping Methods Used:
   • rss        : 33 sources
   • none       : 11 sources
   • html       : 4 sources
   • homepage   : 2 sources

🏆 Top 10 Sources by Article Count:


,source_name,articles_after_filter,method_used,elapsed_seconds
5,IEEE Spectrum,21,rss,1.75
42,Electrek,17,rss,20.18
41,CNBC Tech,13,homepage,61.76
13,TechCrunch,11,rss,16.34
15,Wired,11,rss,22.66
34,CleanTechnica,11,rss,4.65
12,Engadget,10,rss,15.30
40,Semiconductor Engineering,10,rss,18.31
2,VentureBeat,7,rss,1.89
4,Digital Trends,7,rss,2.22



🐢 Slowest Sources:


,source_name,elapsed_seconds,method_used
49,Washington Post Tech,131.87,none
39,CNET,66.78,rss
48,The Atlantic Tech,66.41,rss
47,CNN Tech,62.97,homepage
41,CNBC Tech,61.76,homepage



❌ Failed Sources (21):


TypeError: 'NoneType' object is not subscriptable